# Notebook 2 of 4 — Three tuned models: logistic regression, random forest and LightGBM

In this notebook I train and tune three different kinds of model on the TRAIN part from notebook 1:

| Model | The idea, simply | Why I use it |
|---|---|---|
| Logistic regression | Adds up the features with a weight each, like a points score, and turns the total into a probability | Simple baseline that is very easy to explain |
| Random forest | Asks hundreds of decision trees and averages their votes, like asking a crowd instead of one person | A strong, robust ensemble |
| LightGBM | Builds trees one after another, each one fixing the mistakes of the trees before it, like a student correcting past exam errors | Usually the strongest on table data |

At the end I retrain the best version of each model and save it for notebook 4.

---
## Step 0 — Setup and loading the prepared data

In [2]:
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import joblib
from scipy.stats import randint, uniform, loguniform

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import StratifiedGroupKFold, GridSearchCV, RandomizedSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RESULTS = ROOT / 'results'

d = joblib.load(RESULTS / 'prepared_data.joblib')
X_tr, y_tr, g_tr = d['X_tr'], d['y_tr'], d['g_tr']
X_fit, y_fit = d['X_fit'], d['y_fit']
cat_cols, num_cols = d['cat_cols'], d['num_cols']

---
## Step 2c — Preparing the inputs inside a pipeline

Models need numbers, so I prepare the columns first:
- **Numbers:** I fill any gap with the median and then scale them, so "days in hospital" (1–14) and "lab tests"
  (1–132) are on a similar scale and neither shouts louder just because its numbers are bigger.
- **Categories:** I fill any gap with "Missing" and then **one-hot encode** them. One-hot is like a row of light
  switches, one per possible value, with only the matching switch turned on. For example, "discharged home" turns on
  the `discharge_1` switch and leaves all the others off.
- Categories seen fewer than 50 times are merged into one "infrequent" switch, so the many rare medical specialties do
  not each get a switch the model can barely learn from.

I put all of this inside a **pipeline**, a recipe that is followed in the same order every time. The pipeline matters
for fairness: during cross-validation the scaler and encoder are re-learned on each training fold only, so they never
peek at the rows they are tested on.

In [3]:
def make_preprocessor(cat_cols, num_cols):
    num_pipe = Pipeline([('imp', SimpleImputer(strategy='median')),
                         ('sc', StandardScaler())])
    cat_pipe = Pipeline([('imp', SimpleImputer(strategy='constant', fill_value='Missing')),
                         ('oh', OneHotEncoder(handle_unknown='infrequent_if_exist', min_frequency=50))])
    return ColumnTransformer([('num', num_pipe, num_cols),
                              ('cat', cat_pipe, cat_cols)])

pre = make_preprocessor(cat_cols, num_cols)
print('Inputs after one-hot encoding:', pre.fit(X_tr).transform(X_tr[:5]).shape[1])

Inputs after one-hot encoding: 208


The 48 original columns become about 200 model inputs once every category has its own switch.

---
## Step 3a — How I compare settings fairly: grouped cross-validation

Every model has settings ("hyperparameters"), such as how deep a tree may grow. To pick good settings I use
**3-fold cross-validation**. I cut TRAIN into 3 parts, train on 2 and test on the third, then rotate so every part is
the test once. It is like sitting three mock exams instead of one, so a lucky or unlucky paper does not decide.

`StratifiedGroupKFold` does two things at once:
- each patient stays in one part (no leakage, as in notebook 1),
- each part keeps about 11% readmissions.

I score every setting with three measures:
- **PR-AUC** (my main score): how well the model ranks the rare readmissions above everyone else. Guessing scores
  about 0.11, the share of readmissions.
- **ROC-AUC**: the chance that a random readmitted patient gets a higher risk than a random non-readmitted one.
  0.5 is guessing. Published studies on this data reach about 0.65–0.70.
- **Brier score**: how far the predicted probabilities are from what really happened. Lower is better.

The helper below runs a search, times it and returns one tidy row: the average score over the 3 folds ± how much it
varied.

In [4]:
cv = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=SEED)

scoring = {'ap': 'average_precision', 'roc': 'roc_auc', 'brier': 'neg_brier_score'}

def run_search(search, name, search_type):
    start = time.time()
    search.fit(X_tr, y_tr, groups=g_tr)
    seconds = time.time() - start
    best = pd.DataFrame(search.cv_results_).iloc[search.best_index_]
    return {
        'model': name,
        'search': search_type,
        'settings tried': len(search.cv_results_['params']),
        'seconds': round(seconds),
        'best settings': {k.replace('clf__', ''): (round(float(v), 4) if isinstance(v, float) else v)
                          for k, v in search.best_params_.items()},
        'PR-AUC': f"{best.mean_test_ap:.3f} ± {best.std_test_ap:.3f}",
        'ROC-AUC': f"{best.mean_test_roc:.3f} ± {best.std_test_roc:.3f}",
        'Brier': f"{-best.mean_test_brier:.3f} ± {best.std_test_brier:.3f}",
    }

---
## Step 3b — Logistic regression (grid search)

Logistic regression has one main setting, `C`. It controls how large the weights may grow. A small `C` keeps the model
simple and cautious; a large `C` lets it follow the training data more closely, with the risk of learning noise.
With only one setting I can simply try a fixed list of values. This is a **grid search**.

Because readmissions are rare, I use `class_weight='balanced'`. It makes every readmitted patient count about 8 times
in training, so the model cannot get away with ignoring them.

In [5]:
lr = Pipeline([('pre', make_preprocessor(cat_cols, num_cols)),
               ('clf', LogisticRegression(class_weight='balanced', max_iter=2000))])

lr_search = GridSearchCV(lr, {'clf__C': [0.001, 0.01, 0.1, 1, 10]},
                         cv=cv, scoring=scoring, refit='ap', n_jobs=-1)

res_lr = run_search(lr_search, 'LogReg', 'grid')
pd.DataFrame([res_lr])

,model,search,settings tried,seconds,best settings,PR-AUC,ROC-AUC,Brier
0,LogReg,grid,5,12,{'C': 0.1},0.212 ± 0.002,0.661 ± 0.005,0.226 ± 0.001


---
## Step 3c — LightGBM (random search)

LightGBM has many settings: number of trees, learning rate, tree size and more. A full grid would mean thousands of
runs. Instead I use a **random search**: I try 15 random combinations from sensible ranges. It is like tasting 15
random dishes from a big menu rather than every single one; with good ranges it finds a good dish quickly.

`scale_pos_weight` is LightGBM's version of class weighting. I let the search choose between no weighting (1),
a middle value (√8) and full weighting (about 8).

In [6]:
spw = (y_tr == 0).sum() / (y_tr == 1).sum()

lgb = Pipeline([('pre', make_preprocessor(cat_cols, num_cols)),
                ('clf', LGBMClassifier(random_state=SEED, verbose=-1))])

lgb_space = {
    'clf__n_estimators': randint(200, 800),
    'clf__learning_rate': loguniform(0.01, 0.1),
    'clf__num_leaves': randint(15, 63),
    'clf__min_child_samples': randint(20, 200),
    'clf__subsample': uniform(0.6, 0.4),
    'clf__subsample_freq': [1],
    'clf__colsample_bytree': uniform(0.5, 0.5),
    'clf__scale_pos_weight': [1, spw ** 0.5, spw],
}

lgb_search = RandomizedSearchCV(lgb, lgb_space, n_iter=15, cv=cv, scoring=scoring,
                                refit='ap', random_state=SEED)

res_lgb = run_search(lgb_search, 'LightGBM', 'random')
pd.DataFrame([res_lgb])

,model,search,settings tried,seconds,best settings,PR-AUC,ROC-AUC,Brier
0,LightGBM,random,15,273,"{'colsample_bytree': 0.8926, 'learning_rate': ...",0.228 ± 0.004,0.673 ± 0.006,0.096 ± 0.000


---
## Step 3d — Random forest (random search)

The forest uses 300 trees. I let a random search try 10 combinations of how deep each tree may grow, how many patients
each leaf must hold, and how many features each split may look at. `class_weight='balanced_subsample'` applies class
weighting inside each tree's own sample.

In [7]:
rf = Pipeline([('pre', make_preprocessor(cat_cols, num_cols)),
               ('clf', RandomForestClassifier(n_estimators=300, class_weight='balanced_subsample',
                                              n_jobs=-1, random_state=SEED))])

rf_space = {
    'clf__max_depth': [6, 10, 14, None],
    'clf__min_samples_leaf': randint(5, 100),
    'clf__max_features': ['sqrt', 0.2, 0.4],
}

rf_search = RandomizedSearchCV(rf, rf_space, n_iter=10, cv=cv, scoring=scoring,
                               refit='ap', random_state=SEED)

res_rf = run_search(rf_search, 'RandomForest', 'random')
pd.DataFrame([res_rf])

,model,search,settings tried,seconds,best settings,PR-AUC,ROC-AUC,Brier
0,RandomForest,random,10,128,"{'max_depth': None, 'max_features': 0.2, 'min_...",0.219 ± 0.003,0.669 ± 0.006,0.179 ± 0.001


---
## Step 3e — Comparing the three models

In [8]:
cv_table = pd.DataFrame([res_lr, res_lgb, res_rf])
# I save this table because notebook 4 uses it when choosing the final model
cv_table.to_csv(RESULTS / 'cv_results.csv', index=False)
pd.set_option('display.max_colwidth', 120)
cv_table

,model,search,settings tried,seconds,best settings,PR-AUC,ROC-AUC,Brier
0,LogReg,grid,5,12,{'C': 0.1},0.212 ± 0.002,0.661 ± 0.005,0.226 ± 0.001
1,LightGBM,random,15,273,"{'colsample_bytree': 0.8926, 'learning_rate': 0.0158, 'min_child_samples': 74, 'n_estimators': 443, 'num_leaves': 17...",0.228 ± 0.004,0.673 ± 0.006,0.096 ± 0.000
2,RandomForest,random,10,128,"{'max_depth': None, 'max_features': 0.2, 'min_samples_leaf': 26}",0.219 ± 0.003,0.669 ± 0.006,0.179 ± 0.001


All three models clearly beat guessing (PR-AUC about 0.11): they score roughly 0.21–0.23, about twice the baseline.
ROC-AUC sits around 0.66–0.67, in line with published work on this dataset. The "±" values are small, so the
ranking is stable across the three folds.

LightGBM ranks patients best. Interestingly, its search chose `scale_pos_weight = 1`, meaning **no** class weighting
worked best for it. That is also why its Brier score is much lower: without weighting, its probabilities stay close to
the real 11% rate, while the weighted models push probabilities upwards.

### Retraining the best settings on FIT

Now I retrain each model with its best settings on the FIT part only. I keep VAL untouched on purpose, because in
notebook 4 I use it to calibrate the probabilities and choose the threshold. Using VAL here as well would be like
marking my own homework with the answer sheet I studied from.

In [9]:
final_models = {}
for name, search in [('LogReg', lr_search), ('RandomForest', rf_search), ('LightGBM', lgb_search)]:
    final_models[name] = clone(search.best_estimator_).fit(X_fit, y_fit)

joblib.dump(final_models, RESULTS / 'final_models.joblib')
print('Saved results/final_models.joblib')

Saved results/final_models.joblib
